# Question 3

## I. Derivative Of Activation Functions (5pt)

The following cell contains an implementation of some activation functions. Implement the corresponding derivatives.

In [ ]:
from model import sigmoid, tanh, softmax, d_sigmoid, d_tanh, d_softmax, one_hot, cross_entropy, FullyConnectedNetwork


In [ ]:
# Derivatives are implemented in model.py.


## II. Train a Fully Connected network on MNIST (30pt)

In the following exercise, you will create a classifier for the MNIST dataset.
You should write your own training and evaluation code and meet the following
constraints:
<ul>
<li> You are only allowed to use torch tensor manipulations.</li>
<li> You are NOT allowed to use:
  <ul>
  <li> Auto-differentiation - backward()</li>
  <li> Built-in loss functions</li>
  <li> Built-in activations</li>
  <li> Built-in optimization</li>
  <li> Built-in layers (torch.nn)</li>
  </ul>
  </li>
</ul>
</h4>

a. The required classifier class is defined.

You should implement the forward and backward passes of the model.
Train the model and plot the model's accuracy and loss (both on train and test sets) as a function of the epochs.
You should save the model's weights and biases. Change the student_ids to yours.

In this section, you <b>must</b> use the "set_seed" function with the given seed and <b>sigmoid</b> as an activation function.

In [ ]:
import torchvision
from torch.utils.data import DataLoader

import os
import matplotlib.pyplot as plt

# Constants
SEED = 42
EPOCHS = 16
BATCH_SIZE = 32
NUM_OF_CLASSES = 10

# Setting seed
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


# Transformation for the data
transform = torchvision.transforms.Compose(
    [torchvision.transforms.ToTensor(),
     torch.flatten])




In [ ]:
# Create dataloaders
train_dataset = torchvision.datasets.MNIST(root='./data', train=True,
                                            download=True, transform=transform)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE)


test_dataset = torchvision.datasets.MNIST(root='./data', train=False,
                                           download=True, transform=transform)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE,)

In [ ]:
# Network implementation is imported from model.py.


In [ ]:
set_seed(SEED)
model = FullyConnectedNetwork(784, 10, 128, sigmoid, lr=0.01)

b. Train the model with various learning rates (at least 3).

Plot the model's accuracy and loss (both on train and test sets) as a function of the epochs.
Discuss the differences in training with different learning rates. Support your answer with plots.

In [ ]:
def train_model(model, train_dataloader, test_dataloader, epochs):

    train_loss_history = []
    test_loss_history = []
    train_acc_history = []
    test_acc_history = []

    for epoch in range(epochs):

        #  Training Phase
        epoch_train_loss = 0
        correct_train = 0
        total_train = 0

        for x_batch, y_batch in train_dataloader:
            # Forward pass
            y_hat = model.forward(x_batch)

            # Calculate loss (for logging)
            loss = cross_entropy(y_batch, y_hat)
            epoch_train_loss += loss.item()

            # Backward pass and parameter update
            model.backward(x_batch, y_batch, y_hat)

            # Calculate training accuracy
            predicted = torch.argmax(y_hat, dim=1)
            total_train += y_batch.size(0)
            correct_train += (predicted == y_batch).sum().item()

        avg_train_loss = epoch_train_loss / len(train_dataloader)
        train_accuracy = correct_train / total_train

        train_loss_history.append(avg_train_loss)
        train_acc_history.append(train_accuracy)

        #  Testing Phase
        test_loss, test_accuracy = evaluate_model(model, test_dataloader)
        test_loss_history.append(test_loss)
        test_acc_history.append(test_accuracy)

        print(f"Epoch {epoch+1}/{epochs}: Train Loss={avg_train_loss:.4f}, Train Acc={train_accuracy:.4f}, Test Acc={test_accuracy:.4f}")

    return train_loss_history, test_loss_history, train_acc_history, test_acc_history


def evaluate_model(model, dataloader):
    """
    Evaluates the network on a given dataset.
    """
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            y_hat = model.forward(x_batch)

            # Calculate loss
            loss = cross_entropy(y_batch, y_hat)
            total_loss += loss.item()

            # Calculate accuracy
            predicted = torch.argmax(y_hat, dim=1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
#  Setup
input_size = 784 # 28*28 MNIST image
hidden_size1 = 128
output_size = NUM_OF_CLASSES
activation_func = sigmoid

# Define Learning Rates to Test
LRS = [0.001, 0.1, 1.0]
results = {}

print("Starting training with different learning rates...")

for lr in LRS:
    print(f"\nTraining with Learning Rate (LR): {lr}")

    # Instantiate a fresh model for each LR test
    model = FullyConnectedNetwork(
        input_size,
        output_size,
        hidden_size1,
        activation_func,
        lr=lr
    )

    # Run training
    train_loss, test_loss, train_acc, test_acc = train_model(
        model,
        train_dataloader,
        test_dataloader,
        EPOCHS
    )

    # Store results, including model parameters for later saving
    results[lr] = {
        'train_loss': train_loss,
        'test_loss': test_loss,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'W1': model.W1.clone(),
        'b1': model.b1.clone(),
        'W2': model.W2.clone(),
        'b2': model.b2.clone(),
    }

print("\nTraining complete!")

# Find the best model based on test accuracy
best_lr = None
best_test_acc = -1.0
best_model_params = None

for lr, data in results.items():
    # Compare with the final test accuracy of the current learning rate
    if data['test_acc'][-1] > best_test_acc:
        best_test_acc = data['test_acc'][-1]
        best_lr = lr
        best_model_params = {
            'W1': data['W1'],
            'b1': data['b1'],
            'W2': data['W2'],
            'b2': data['b2']
        }

print(f"\nBest model found with Learning Rate: {best_lr} and Test Accuracy: {best_test_acc:.4f}")


In [ ]:
epochs_list = range(1, EPOCHS + 1)

plt.figure(figsize=(16, 6))

# --- Loss Plot (Train and Test) ---
plt.subplot(1, 2, 1)
for lr, data in results.items():
    plt.plot(epochs_list, data['train_loss'], label=f'Train LR={lr}', linestyle='-')
    plt.plot(epochs_list, data['test_loss'], label=f'Test LR={lr}', linestyle='--')

plt.title('Loss (Cross-Entropy) vs. Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)


# --- Accuracy Plot (Train and Test) ---
plt.subplot(1, 2, 2)
for lr, data in results.items():
    plt.plot(epochs_list, data['train_acc'], label=f'Train LR={lr}', linestyle='-')
    plt.plot(epochs_list, data['test_acc'], label=f'Test LR={lr}', linestyle='--')

plt.title('Accuracy vs. Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.show()

#Saving the model

In [ ]:
students_ids = "000000000_000000000"
# Save the parameters of the best model
torch.save(best_model_params, f"HW1_{students_ids}.pkl")
print(f"Best model parameters saved to HW1_{students_ids}.pkl")


#Results:

### As we can see from the graphs and the accuracies, learning rate of 1.0 was the best of the three.
### 0.1 and 0.001 will converge eventually, but it would take more time, because they learn slower.